In [44]:
import pandas as pd
import numpy as np

from scipy.stats import linregress

In [ ]:
nav_history = pd.read_csv(
    "../data/processed/02_nav_history_calendar.csv"
)

performance = pd.read_csv(
    "../data/processed/07_scheme_performance.csv"
)

benchmark = pd.read_csv(
    "../data/processed/10_benchmark_indices.csv"
)

In [46]:
nav_history["date"] = pd.to_datetime(
    nav_history["date"]
)

benchmark["date"] = pd.to_datetime(
    benchmark["date"]
)

In [47]:
nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)

nav_history["daily_return"] = (
    nav_history.groupby("amfi_code")["nav"]
    .pct_change()
)

In [48]:
nav_history.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [94]:
cagr_results = []

for amfi_code, group in nav_history.groupby("amfi_code"):

    group = group.sort_values("date")

    start_nav = group.iloc[0]["nav"]
    end_nav = group.iloc[-1]["nav"]

    trading_days = len(group)
    years = trading_days / 252

    cagr = (
        (end_nav / start_nav)
        ** (1 / years)
        - 1
    ) * 100

    cagr_results.append(
        [amfi_code, cagr]
    )

cagr_df = pd.DataFrame(
    cagr_results,
    columns=["amfi_code", "cagr_pct"]
)

cagr_df.head()

,amfi_code,cagr_pct
0,100016,2.541248
1,100025,4.294835
2,100033,28.899384
3,101206,22.604787
4,101207,7.643258


In [50]:
risk_free_rate = 0.065

sharpe_results = []

for amfi_code, group in nav_history.groupby("amfi_code"):

    returns = group["daily_return"].dropna()

    annual_return = returns.mean() * 252

    annual_volatility = returns.std() * np.sqrt(252)

    sharpe = (
        annual_return - risk_free_rate
    ) / annual_volatility

    sharpe_results.append(
        [amfi_code, sharpe]
    )

sharpe_df = pd.DataFrame(
    sharpe_results,
    columns=["amfi_code", "sharpe_ratio_calc"]
)

sharpe_df.head()

,amfi_code,sharpe_ratio_calc
0,100016,-0.201517
1,100025,-0.567095
2,100033,1.093699
3,101206,1.027213
4,101207,0.162661


In [51]:
sortino_results = []

for amfi_code, group in nav_history.groupby("amfi_code"):

    returns = group["daily_return"].dropna()

    annual_return = returns.mean() * 252

    downside_returns = returns[returns < 0]

    downside_std = (
        downside_returns.std()
        * np.sqrt(252)
    )

    sortino = (
        annual_return - risk_free_rate
    ) / downside_std

    sortino_results.append(
        [amfi_code, sortino]
    )

sortino_df = pd.DataFrame(
    sortino_results,
    columns=["amfi_code", "sortino_ratio_calc"]
)

sortino_df.head()

,amfi_code,sortino_ratio_calc
0,100016,-0.351047
1,100025,-0.941821
2,100033,1.829134
3,101206,1.799563
4,101207,0.276644


In [52]:
def max_drawdown(nav_series):

    rolling_max = nav_series.cummax()

    drawdown = (
        nav_series /
        rolling_max
        - 1
    )

    return drawdown.min()

In [53]:
mdd_results = []

for amfi_code, group in nav_history.groupby("amfi_code"):

    group = group.sort_values("date")

    mdd = max_drawdown(
        group["nav"]
    )

    mdd_results.append(
        [amfi_code, mdd * 100]
    )

mdd_df = pd.DataFrame(
    mdd_results,
    columns=[
        "amfi_code",
        "max_drawdown_pct_calc"
    ]
)

mdd_df.head()

,amfi_code,max_drawdown_pct_calc
0,100016,-24.734441
1,100025,-4.308264
2,100033,-16.217209
3,101206,-11.291596
4,101207,-35.446916


In [54]:
performance_metrics = (
    cagr_df
    .merge(
        sharpe_df,
        on="amfi_code"
    )
    .merge(
        sortino_df,
        on="amfi_code"
    )
    .merge(
        mdd_df,
        on="amfi_code"
    )
)

performance_metrics.head()

,amfi_code,cagr_pct,sharpe_ratio_calc,sortino_ratio_calc,max_drawdown_pct_calc
0,100016,2.635246,-0.201517,-0.351047,-24.734441
1,100025,4.455091,-0.567095,-0.941821,-4.308264
2,100033,30.099704,1.093699,1.829134,-16.217209
3,101206,23.520489,1.027213,1.799563,-11.291596
4,101207,7.933121,0.162661,0.276644,-35.446916


In [55]:
performance_metrics.to_csv(
    "../data/processed/performance_metrics.csv",
    index=False
)

print(
    "Saved performance_metrics.csv"
)

Saved performance_metrics.csv


In [56]:
benchmark.columns.tolist()

['date', 'index_name', 'close_value']

In [57]:
benchmark = benchmark.sort_values(
    ["index_name", "date"]
)

benchmark["benchmark_return"] = (
    benchmark.groupby("index_name")
    ["close_value"]
    .pct_change()
)

benchmark.head()

,date,index_name,close_value,benchmark_return
3450,2022-01-03,BSE_SMALLCAP,26554.60,NaN
3451,2022-01-04,BSE_SMALLCAP,27079.92,0.019783
3452,2022-01-05,BSE_SMALLCAP,27313.35,0.008620
3453,2022-01-06,BSE_SMALLCAP,27377.05,0.002332
3454,2022-01-07,BSE_SMALLCAP,26316.86,-0.038726


In [58]:
benchmark["index_name"].unique()

<StringArray>
[   'BSE_SMALLCAP',     'CRISIL_GILT',   'CRISIL_LIQUID',        'NIFTY100',
         'NIFTY50',        'NIFTY500', 'NIFTY_MIDCAP150']
Length: 7, dtype: str

In [59]:
nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

nifty100 = nifty100[
    ["date", "benchmark_return"]
]

nifty100.head()

,date,benchmark_return
1150,2022-01-03,NaN
1151,2022-01-04,-0.013540
1152,2022-01-05,0.004003
1153,2022-01-06,-0.002935
1154,2022-01-07,0.006150


In [60]:
fund_returns = nav_history[
    ["amfi_code", "date", "daily_return"]
].copy()

fund_returns.head()

,amfi_code,date,daily_return
5750,100016,2022-01-03,NaN
5751,100016,2022-01-04,-0.010306
5752,100016,2022-01-05,0.012865
5753,100016,2022-01-06,-0.011377
5754,100016,2022-01-07,-0.001210


In [61]:
fund_returns.shape

(46000, 3)

In [62]:
from scipy.stats import linregress

alpha_beta_results = []

for amfi_code, group in fund_returns.groupby("amfi_code"):

    merged = pd.merge(
        group,
        nifty100,
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    if len(merged) < 50:
        continue

    slope, intercept, r, p, stderr = linregress(
        merged["benchmark_return"],
        merged["daily_return"]
    )

    beta = slope

    alpha = intercept * 252 * 100

    alpha_beta_results.append(
        [amfi_code, alpha, beta]
    )

alpha_beta_df = pd.DataFrame(
    alpha_beta_results,
    columns=[
        "amfi_code",
        "alpha",
        "beta"
    ]
)

alpha_beta_df.head()

,amfi_code,alpha,beta
0,100016,3.747581,-0.058268
1,100025,4.281792,0.001158
2,100033,27.195355,0.005104
3,101206,21.399785,0.021086
4,101207,10.897092,-0.065289


In [63]:
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)

print("Saved alpha_beta.csv")

Saved alpha_beta.csv


In [64]:
scorecard = performance_metrics.copy()

expense = performance[
    ["amfi_code", "expense_ratio_pct"]
]

scorecard = scorecard.merge(
    expense,
    on="amfi_code"
)

scorecard["cagr_rank"] = (
    scorecard["cagr_pct"]
    .rank(pct=True)
)

scorecard["sharpe_rank"] = (
    scorecard["sharpe_ratio_calc"]
    .rank(pct=True)
)

scorecard["mdd_rank"] = (
    scorecard["max_drawdown_pct_calc"]
    .rank(
        pct=True,
        ascending=False
    )
)

scorecard["expense_rank"] = (
    scorecard["expense_ratio_pct"]
    .rank(
        pct=True,
        ascending=False
    )
)

scorecard.head()

,amfi_code,cagr_pct,sharpe_ratio_calc,sortino_ratio_calc,max_drawdown_pct_calc,expense_ratio_pct,cagr_rank,sharpe_rank,mdd_rank,expense_rank
0,100016,2.635246,-0.201517,-0.351047,-24.734441,1.55,0.100,0.150,0.850,0.2250
1,100025,4.455091,-0.567095,-0.941821,-4.308264,0.56,0.125,0.050,0.100,0.9750
2,100033,30.099704,1.093699,1.829134,-16.217209,1.38,0.850,0.850,0.500,0.6000
3,101206,23.520489,1.027213,1.799563,-11.291596,1.60,0.725,0.800,0.225,0.1125
4,101207,7.933121,0.162661,0.276644,-35.446916,1.53,0.350,0.325,0.950,0.3125


In [65]:
scorecard = scorecard.merge(
    alpha_beta_df,
    on="amfi_code"
)

scorecard.head()

,amfi_code,cagr_pct,sharpe_ratio_calc,sortino_ratio_calc,max_drawdown_pct_calc,expense_ratio_pct,cagr_rank,sharpe_rank,mdd_rank,expense_rank,alpha,beta
0,100016,2.635246,-0.201517,-0.351047,-24.734441,1.55,0.100,0.150,0.850,0.2250,3.747581,-0.058268
1,100025,4.455091,-0.567095,-0.941821,-4.308264,0.56,0.125,0.050,0.100,0.9750,4.281792,0.001158
2,100033,30.099704,1.093699,1.829134,-16.217209,1.38,0.850,0.850,0.500,0.6000,27.195355,0.005104
3,101206,23.520489,1.027213,1.799563,-11.291596,1.60,0.725,0.800,0.225,0.1125,21.399785,0.021086
4,101207,7.933121,0.162661,0.276644,-35.446916,1.53,0.350,0.325,0.950,0.3125,10.897092,-0.065289


In [66]:
scorecard["alpha_rank"] = (
    scorecard["alpha"]
    .rank(pct=True)
)

In [67]:
scorecard["fund_score"] = (
      0.30 * scorecard["cagr_rank"]
    + 0.25 * scorecard["sharpe_rank"]
    + 0.20 * scorecard["alpha_rank"]
    + 0.15 * scorecard["expense_rank"]
    + 0.10 * scorecard["mdd_rank"]
) * 100

In [68]:
top10 = (
    scorecard
    .sort_values(
        "fund_score",
        ascending=False
    )
    .head(10)
)

top10[
    [
        "amfi_code",
        "fund_score",
        "alpha",
        "beta"
    ]
]

,amfi_code,fund_score,alpha,beta
25,120505,87.5000,29.263583,0.000549
21,119598,82.6250,30.336965,-0.023196
39,149324,80.1875,30.057878,0.011455
30,120843,79.0000,27.330465,-0.022830
2,100033,78.2500,27.195355,0.005104
34,148567,77.7500,26.983751,0.023684
36,148569,76.9375,28.270368,0.018134
16,119094,74.7500,26.076669,-0.066265
9,118632,70.0625,21.829398,-0.008354
19,119551,70.0625,23.201007,-0.031751


In [69]:
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print("Saved fund_scorecard.csv")

Saved fund_scorecard.csv


In [70]:
fund_master = pd.read_csv(
    "../data/processed/01_fund_master.csv"
)

fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [71]:
fund_master.columns.tolist()

['amfi_code',
 'fund_house',
 'scheme_name',
 'category',
 'sub_category',
 'plan',
 'launch_date',
 'benchmark',
 'expense_ratio_pct',
 'exit_load_pct',
 'min_sip_amount',
 'min_lumpsum_amount',
 'fund_manager',
 'risk_category',
 'sebi_category_code']

In [72]:
performance.columns.tolist()

['amfi_code',
 'scheme_name',
 'fund_house',
 'category',
 'plan',
 'return_1yr_pct',
 'return_3yr_pct',
 'return_5yr_pct',
 'benchmark_3yr_pct',
 'alpha',
 'beta',
 'sharpe_ratio',
 'sortino_ratio',
 'std_dev_ann_pct',
 'max_drawdown_pct',
 'aum_crore',
 'expense_ratio_pct',
 'morningstar_rating',
 'risk_grade']

In [73]:
scorecard = performance[
    [
        "amfi_code",
        "scheme_name",
        "fund_house",
        "category",
        "return_3yr_pct",
        "alpha",
        "beta",
        "sharpe_ratio",
        "sortino_ratio",
        "max_drawdown_pct",
        "expense_ratio_pct",
        "risk_grade"
    ]
].copy()

In [74]:
scorecard["return_rank"] = scorecard["return_3yr_pct"].rank(
    ascending=False
)

scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(
    ascending=False
)

scorecard["alpha_rank"] = scorecard["alpha"].rank(
    ascending=False
)

In [75]:
scorecard["expense_rank"] = scorecard["expense_ratio_pct"].rank(
    ascending=True
)

scorecard["beta_rank"] = scorecard["beta"].rank(
    ascending=True
)

scorecard["drawdown_rank"] = scorecard["max_drawdown_pct"].rank(
    ascending=False
)

In [76]:
scorecard["fund_score"] = (
    scorecard["return_rank"] * 0.30 +
    scorecard["sharpe_rank"] * 0.25 +
    scorecard["alpha_rank"] * 0.20 +
    scorecard["expense_rank"] * 0.10 +
    scorecard["beta_rank"] * 0.05 +
    scorecard["drawdown_rank"] * 0.10
)

In [77]:
scorecard = scorecard.sort_values(
    "fund_score",
    ascending=False
)

In [78]:
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print("fund_scorecard.csv saved")

fund_scorecard.csv saved


In [79]:
scorecard[
    [
        "scheme_name",
        "fund_house",
        "category",
        "fund_score"
    ]
].head(10)

,scheme_name,fund_house,category,fund_score
36,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset MF,ELSS,31.875
10,ICICI Pru Bluechip Fund - Regular - Growth,ICICI Prudential MF,Large Cap,31.800
24,Axis Bluechip Fund - Regular - Growth,Axis Mutual Fund,Large Cap,28.600
0,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,26.550
8,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,26.075
31,UTI Nifty 50 Index Fund - Regular - Growth,UTI Mutual Fund,Index,26.075
20,Kotak Bluechip Fund - Regular - Growth,Kotak Mahindra MF,Large Cap,25.650
17,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,25.425
1,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,24.550
26,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,Mid Cap,24.375


In [80]:
import os

os.path.exists(
    "../data/processed/fund_scorecard.csv"
)

True

In [81]:
scorecard.columns.tolist()

['amfi_code',
 'scheme_name',
 'fund_house',
 'category',
 'return_3yr_pct',
 'alpha',
 'beta',
 'sharpe_ratio',
 'sortino_ratio',
 'max_drawdown_pct',
 'expense_ratio_pct',
 'risk_grade',
 'return_rank',
 'sharpe_rank',
 'alpha_rank',
 'expense_rank',
 'beta_rank',
 'drawdown_rank',
 'fund_score']

In [82]:
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print(scorecard.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'return_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown_pct', 'expense_ratio_pct', 'risk_grade', 'return_rank', 'sharpe_rank', 'alpha_rank', 'expense_rank', 'beta_rank', 'drawdown_rank', 'fund_score']


In [83]:
saved = pd.read_csv(
    "../data/processed/fund_scorecard.csv"
)

saved.columns.tolist()

['amfi_code',
 'scheme_name',
 'fund_house',
 'category',
 'return_3yr_pct',
 'alpha',
 'beta',
 'sharpe_ratio',
 'sortino_ratio',
 'max_drawdown_pct',
 'expense_ratio_pct',
 'risk_grade',
 'return_rank',
 'sharpe_rank',
 'alpha_rank',
 'expense_rank',
 'beta_rank',
 'drawdown_rank',
 'fund_score']

In [ ]:
import pandas as pd

nav = pd.read_csv("../data/processed/02_nav_history_calendar.csv")

nav["date"] = pd.to_datetime(nav["date"])

print("Min date:", nav["date"].min())
print("Max date:", nav["date"].max())

total_days = (
    nav["date"].max() - nav["date"].min()
).days + 1

actual_days = nav["date"].nunique()

print("Calendar days:", total_days)
print("Actual dates:", actual_days)
print("Missing dates:", total_days - actual_days)

Min date: 2022-01-03 00:00:00
Max date: 2026-05-29 00:00:00
Calendar days: 1608
Actual dates: 1150
Missing dates: 458


In [ ]:
import pandas as pd

nav = pd.read_csv("../data/processed/02_nav_history_calendar.csv")

nav["date"] = pd.to_datetime(nav["date"])

print("Before:", nav.shape)
nav.head()

Before: (46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [87]:
full_nav = []

for amfi_code, group in nav.groupby("amfi_code"):

    group = group.sort_values("date")

    full_dates = pd.date_range(
        start=group["date"].min(),
        end=group["date"].max(),
        freq="D"
    )

    group = (
        group.set_index("date")
             .reindex(full_dates)
    )

    group["amfi_code"] = amfi_code

    group["nav"] = group["nav"].ffill()

    group = (
        group.reset_index()
             .rename(columns={"index":"date"})
    )

    full_nav.append(group)

nav_full = pd.concat(
    full_nav,
    ignore_index=True
)

print("After:", nav_full.shape)
nav_full.head()

After: (64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [88]:
total_days = (
    nav_full["date"].max() -
    nav_full["date"].min()
).days + 1

actual_days = nav_full["date"].nunique()

print("Calendar days:", total_days)
print("Actual dates:", actual_days)
print("Missing dates:", total_days - actual_days)

Calendar days: 1608
Actual dates: 1608
Missing dates: 0


In [89]:
nav_full.to_csv(
    "../data/processed/02_nav_history_calendar.csv",
    index=False
)

In [90]:
nav_full

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639
...,...,...,...
64315,2026-05-25,149324,292.4810
64316,2026-05-26,149324,291.2707
64317,2026-05-27,149324,288.8007
64318,2026-05-28,149324,280.6873
